In [12]:
# program arguments
YEAR = "2024"
MONTH = "1"

IGNORE_MISSING = False

In [13]:
# program variables
MONTH_FMT = MONTH.zfill(2)

FILE_PREFIX = f"{YEAR}{MONTH_FMT}"
FILE_NAME = f"{FILE_PREFIX}-citibike-tripdata.zip"
FILE_URL = f"https://s3.amazonaws.com/tripdata/{FILE_NAME}"


BUCKET_NAME = "de_citibike_bucket"
GCS_FILE_URL = f"{YEAR}/{FILE_NAME}"

In [14]:
from typing import Union

import requests
from requests import Response


def _validate_file_contents(r: Response) -> Union[Response, None]:
    """Validates a http response for file existence.

    Returns the requests Response object if file contents not empty.
    If IGNORE_MISSING is True, skip current file and return None.

    Raises exception if file is missing and IGNORE_MISSING is False.
    """

    print("[INFO] Validating remote file and file contents.")

    if r.headers["content-length"] and int(r.headers["content-length"]) > 0:
        return r

    if IGNORE_MISSING:
        print("f[INFO] The remote file is missing or empty. Skip download.")
        return None
    else:
        raise ValueError(f"Remote file is missing or empty.")

In [15]:
# setup env vars
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()

# generate path to gcp account api keys json file
ROOT = Path().resolve().parents[0]
gcp_cred_path = ROOT / os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(gcp_cred_path)

In [ ]:
from google.cloud import storage
from google.cloud.exceptions import Conflict

# TODO - remove bucket_name, re-use BUCKET_NAME (when dev done)
# bucket_name = "de_citibike_bucket"
bucket_name = "de_course_bucket"

storage_client = storage.Client()

# sample GS url: gs://de_course_bucket/taxi/green_tripdata_2019-04.csv


def _file_exists_in_storage(file_name: str, client: storage.Client) -> bool:
    """Verifies if the file exists in the google storage.

    Returns True if the file already exists
    """

    # bucket = client.get_bucket(BUCKET_NAME)
    bucket = client.get_bucket(bucket_name)

    blobs = bucket.list_blobs()

    for blob in blobs:
        if blob.name == file_name:
            print(f"blob.size: {blob.size}")
            return True

    return False

# def _file_exists_in_storage(file_name: str, client: storage.Client) -> bool:
#     return True


# remote file size: 369035302
_file_exists_in_storage("taxi/green_tripdata_2019-04.csv", storage_client)
# _file_exists_in_storage(GCS_FILE_URL, storage_client)

blob.size: 47348207


True

In [17]:
# Pseudo Code

# NOTE: assume bucket exists. It's managed by terraform

# check file exists on internet
# - yes -> next
# - no -> skip -> log
# check file exists in GCS
# - yes -> check file checksum (crc32c - preferable, md5 - also good)
# - -   checksum the same -> skip -> log
# - -   checksum different -> upload to GCS (replace) -> log
# - no -> upload to GCS
# done

def ingest(from_url: str, to_bucket: str):
    """Extract dataset files from_url and upload these to a cloud storage.

    First it checks file exists at the given from_url.
    If success, next checks if the same file exists in the cloud storage.
    It doesn't load and existing file twice to the same storage.
    Finally it upload the file to the storage if this is new.
    """

    with requests.get(from_url, stream=True) as r:
        r.raise_for_status()
        for (k, v) in r.headers.items():
            print(f"k: {k} - v: {v}")

        http_response = _validate_file_contents(r)
        crc_hash = None
        storage_file = _file_exists_in_storage(GCS_FILE_URL, storage_client)
        if http_response and storage_file:
            print("hooray")


ingest(FILE_URL, "blah")

k: x-amz-id-2 - v: p+yeEma38ev+pCBcb9ZBvegL94x9tdImhsmyraDNVDv1XVnV+J51ERcwxBKvV1SGlinpKD14ty4=
k: x-amz-request-id - v: XDQQT121YPWMPRS3
k: Date - v: Thu, 19 Mar 2026 11:56:08 GMT
k: Last-Modified - v: Thu, 03 Jul 2025 15:01:20 GMT
k: ETag - v: "ce6c041ebab0621c1951517b1150eb69-22"
k: x-amz-server-side-encryption - v: AES256
k: Accept-Ranges - v: bytes
k: Content-Type - v: application/zip
k: Content-Length - v: 369035302
k: Server - v: AmazonS3
[INFO] Validating remote file and file contents.
hooray
